In [ ]:
# ============================================================
# PARÁMETROS DEL NOTEBOOK
# ============================================================

LAKEHOUSE = "lh_tinkle_financiero"
ESQUEMA = "200"

# No existe vista vw_Export_Agings: se lee directamente de las tablas.
TABLA_CLIENTES = "Aging_Clientes"
TABLA_PROVEEDORES = "Aging_Proveedores"

# Coincide con la carpeta 'Ageing' ya existente en SharePoint (2. Salida de ficheros),
# así la actividad Copy Data del pipeline puede replicar el patrón de Balance sin más cambios.
CARPETA_SALIDA = "Files/Exportaciones/Ageing"

SOBRESCRIBIR = True
INCLUIR_TIMESTAMP = True

# Excluye facturas ya saldadas (pendiente/vencido = 0).
EXCLUIR_PENDIENTE_CERO = False

print("Parámetros cargados.")

In [ ]:
# ============================================================
# IMPORTACIONES
# ============================================================

from datetime import datetime
import re
import uuid

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

try:
    fs = notebookutils.fs
except NameError:
    from notebookutils import mssparkutils
    fs = mssparkutils.fs

print("Importaciones completadas correctamente.")

In [ ]:
# ============================================================
# ESTRUCTURA ESPERADA DE CADA FUENTE
# ============================================================
# La columna 'Proveedor' de Aging_Clientes es en realidad la sociedad
# emisora (la tabla reutiliza una plantilla de exportación pensada
# originalmente para cuentas a pagar). 'Compañía' cumple ese papel
# en Aging_Proveedores.

FUENTES_AGING = {
    "Clientes": {
        "tabla": TABLA_CLIENTES,
        "columna_sociedad": "Proveedor",
        "columna_importe_pendiente": "S_Pendiente",
        "columnas": [
            "Proveedor",
            "Cobrador",
            "Interno",
            "Nombre Cliente",
            "Pais",
            "Nº Factura",
            "F. Envío  Presup.  Medio  Producto  F. Ins.",
            "NIF",
            "Tipo Factura",
            "F.Factura",
            "F. Vto.",
            "Imp_Factura",
            "S_Pendiente",
        ],
    },
    "Proveedores": {
        "tabla": TABLA_PROVEEDORES,
        "columna_sociedad": "Compañía",
        "columna_importe_pendiente": "Imp_Pendiente",
        "columnas": [
            "Compañía",
            "Cod. Proveedor",
            "Proveedor",
            "CIF Proveedor",
            "Nº Factura",
            "Fecha Factura",
            "Fecha Vencimiento",
            "Fecha Contable",
            "Medio",
            "Moneda",
            "Método",
            "IBAN",
            "Imp_Factura",
            "Imp_Pendiente",
            "Imp_Vencido",
        ],
    },
}

for nombre, definicion in FUENTES_AGING.items():
    print(f"{nombre}: {len(definicion['columnas'])} columnas esperadas")

In [ ]:
# ============================================================
# LECTURA, VALIDACIÓN Y NORMALIZACIÓN DE CADA FUENTE
# ============================================================

def leer_y_preparar_fuente(nombre_fuente: str, definicion: dict) -> DataFrame:
    nombre_completo = f"{LAKEHOUSE}.{ESQUEMA}.{definicion['tabla']}"
    print(f"Leyendo {nombre_fuente}: {nombre_completo}")

    df_origen = spark.read.synapsesql(nombre_completo)
    print(f"Filas {nombre_fuente}: {df_origen.count():,}")

    columnas_faltantes = [
        columna for columna in definicion["columnas"]
        if columna not in df_origen.columns
    ]
    if columnas_faltantes:
        raise ValueError(
            f"{nombre_fuente}: faltan columnas esperadas: {columnas_faltantes}"
        )

    df = df_origen.select(*definicion["columnas"])

    columna_sociedad = definicion["columna_sociedad"]
    columna_pendiente = definicion["columna_importe_pendiente"]

    # Campos obligatorios: sociedad, nº factura y el importe pendiente.
    condicion_vacios = (
        F.col(columna_sociedad).isNull()
        | (F.trim(F.col(columna_sociedad)) == "")
        | F.col("Nº Factura").isNull()
        | (F.trim(F.col("Nº Factura")) == "")
        | F.col(columna_pendiente).isNull()
    )

    errores = df.filter(condicion_vacios)
    if errores.limit(1).count() > 0:
        print(f"{nombre_fuente}: registros con campos obligatorios vacíos:")
        errores.show(20, truncate=False)
        raise ValueError(f"{nombre_fuente}: existen registros con campos obligatorios vacíos.")

    # Aviso (no bloqueante) de facturas duplicadas por sociedad + nº factura.
    duplicados = (
        df.groupBy(columna_sociedad, "Nº Factura")
        .count()
        .filter(F.col("count") > 1)
    )
    if duplicados.limit(1).count() > 0:
        print(f"{nombre_fuente}: aviso, hay nº de factura repetidos dentro de la misma sociedad:")
        duplicados.show(20, truncate=False)

    if EXCLUIR_PENDIENTE_CERO:
        df = df.filter(F.col(columna_pendiente) != 0)

    df = df.orderBy(columna_sociedad, "Nº Factura")

    numero_filas = df.count()
    if numero_filas == 0:
        raise ValueError(f"{nombre_fuente}: no hay registros para exportar.")

    print(f"{nombre_fuente}: filas preparadas: {numero_filas:,}")
    return df


dataframes_aging = {
    nombre: leer_y_preparar_fuente(nombre, definicion)
    for nombre, definicion in FUENTES_AGING.items()
}

In [ ]:
# ============================================================
# FUNCIONES DE GENERACIÓN
# ============================================================

TIMESTAMP_EJECUCION = datetime.now().strftime("%Y%m%d_%H%M%S")


def limpiar_nombre(valor: str) -> str:
    valor = str(valor).strip()
    valor = re.sub(r"[^A-Za-z0-9_-]+", "_", valor)
    valor = re.sub(r"_+", "_", valor)
    return valor.strip("_")


def generar_nombre_ageing(tipo: str, sociedad: str) -> str:
    partes = [limpiar_nombre(sociedad), "Ageing", tipo]
    if INCLUIR_TIMESTAMP:
        partes.append(TIMESTAMP_EJECUCION)
    return "_".join(partes) + ".txt"


def _mover_part_file(ruta_temporal: str, ruta_final: str, extension: str) -> None:
    archivos_temporales = fs.ls(ruta_temporal)

    ficheros_part = [
        archivo for archivo in archivos_temporales
        if archivo.name.startswith("part-") and archivo.name.endswith(extension)
    ]

    if len(ficheros_part) != 1:
        nombres = [archivo.name for archivo in archivos_temporales]
        fs.rm(ruta_temporal, recurse=True)
        raise RuntimeError(
            f"No se ha localizado exactamente un part-file. Contenido encontrado: {nombres}"
        )

    if fs.exists(ruta_final):
        if SOBRESCRIBIR:
            fs.rm(ruta_final, recurse=False)
        else:
            fs.rm(ruta_temporal, recurse=True)
            raise FileExistsError(f"El fichero ya existe: {ruta_final}")

    resultado_movimiento = fs.mv(
        ficheros_part[0].path,
        ruta_final,
        create_path=True,
        overwrite=SOBRESCRIBIR,
    )

    if resultado_movimiento is False:
        fs.rm(ruta_temporal, recurse=True)
        raise RuntimeError(f"No se ha podido mover el fichero a {ruta_final}")

    fs.rm(ruta_temporal, recurse=True)

    if not fs.exists(ruta_final):
        raise RuntimeError(f"El fichero no existe después de moverlo: {ruta_final}")


def escribir_csv(dataframe: DataFrame, ruta_final: str) -> dict:
    """Escribe un TXT separado por ; con cabecera, igual que el Balance."""
    identificador_temporal = uuid.uuid4().hex
    carpeta_destino = ruta_final.rsplit("/", 1)[0]
    ruta_temporal = f"{carpeta_destino}/_temp/{identificador_temporal}"

    fs.mkdirs(carpeta_destino)

    (
        dataframe
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .option("delimiter", ";")
        .option("encoding", "UTF-8")
        .option("nullValue", "")
        .option("emptyValue", "")
        .option("quote", "\u0000")
        .option("escape", "\u0000")
        .option("lineSep", "\n")
        .csv(ruta_temporal)
    )

    _mover_part_file(ruta_temporal, ruta_final, ".csv")

    return {"ruta": ruta_final, "filas": dataframe.count()}

In [ ]:
# ============================================================
# GENERACIÓN DE FICHEROS: 1 POR SOCIEDAD Y POR FUENTE
# ============================================================

resultados = []

for nombre_fuente, definicion in FUENTES_AGING.items():
    df_fuente = dataframes_aging[nombre_fuente]
    columna_sociedad = definicion["columna_sociedad"]

    sociedades = (
        df_fuente.select(columna_sociedad)
        .distinct()
        .orderBy(columna_sociedad)
        .collect()
    )

    print(f"{nombre_fuente}: {len(sociedades)} sociedades a generar")

    for fila_sociedad in sociedades:
        sociedad = fila_sociedad[columna_sociedad]

        df_sociedad = (
            df_fuente
            .filter(F.col(columna_sociedad) == sociedad)
            .orderBy("Nº Factura")
        )

        nombre_fichero = generar_nombre_ageing(nombre_fuente, sociedad)
        ruta_final = f"{CARPETA_SALIDA}/{nombre_fuente}/{nombre_fichero}"

        print(f"Generando Ageing {nombre_fuente} [{sociedad}]: {ruta_final}")

        resultado = escribir_csv(df_sociedad, ruta_final)
        resultado.update({
            "tipo": f"Ageing_{nombre_fuente}",
            "sociedad": sociedad,
            "fichero": nombre_fichero,
        })
        resultados.append(resultado)

print(f"Total de ficheros generados: {len(resultados)}")

In [ ]:
# ============================================================
# RESUMEN DE EJECUCIÓN
# ============================================================

print("")
print("=" * 100)
print("FICHEROS GENERADOS")
print("=" * 100)

for resultado in resultados:
    print(
        f"Tipo: {resultado['tipo']} | "
        f"Sociedad: {resultado['sociedad']} | "
        f"Filas: {resultado['filas']:,}"
    )
    print(f"Ruta: {resultado['ruta']}")
    print("-" * 100)

print(f"TOTAL: {len(resultados)} fichero(s)")

In [ ]:
# ============================================================
# RESULTADO PARA EL PIPELINE
# ============================================================

import json

resultado_pipeline = {
    "estado": "OK",
    "numero_ficheros": len(resultados),
    "numero_ageing_clientes": len([r for r in resultados if r["tipo"] == "Ageing_Clientes"]),
    "numero_ageing_proveedores": len([r for r in resultados if r["tipo"] == "Ageing_Proveedores"]),
    "ficheros": resultados,
    "fecha_ejecucion": datetime.now().isoformat(),
}

resultado_json = json.dumps(resultado_pipeline, ensure_ascii=False)
print(resultado_json)

try:
    notebookutils.notebook.exit(resultado_json)
except NameError:
    try:
        mssparkutils.notebook.exit(resultado_json)
    except Exception:
        pass